# PM10 — Notebook 3: اعتبارسنجی متقابل فضایی–زمانی  


- `MODEL_INDEX=0`: مدل پیشنهادی FCSN با همبستگی مترن
- `MODEL_INDEX=1`: مدل گاوسی با همبستگی مترن
- `MODEL_INDEX=2`: مدل FCSN با همبستگی نمایی



In [ ]:
# Kaggle: Internet must be ON only if pyro is not already available.
!pip -q install pyro-ppl openpyxl

In [ ]:
import os, glob, pickle, shutil
from pathlib import Path

# ==================================================
MODEL_INDEX = 0  # 0: proposed_matern_fscsn | 1: gaussian | 2: skew_exponential
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
TEST_MODE = True   # 
SKIP_EXISTING = True
SEED = 1405

MODELS = ['proposed_matern_fscsn', 'gaussian', 'skew_exponential']
if MODEL_INDEX not in range(len(MODELS)):
    raise ValueError('MODEL_INDEX باید یکی از مقادیر 0، 1 یا 2 باشد.')
if any(f not in range(5) for f in FOLDS_TO_RUN):
    raise ValueError('شماره foldها باید از 0 تا 4 باشد.')
MODEL_NAME = MODELS[MODEL_INDEX]
print('MODEL =', MODEL_NAME)
print('FOLDS =', FOLDS_TO_RUN)
print('TEST_MODE =', TEST_MODE)

BASE_OUT = Path('/kaggle/working/PM10_model_comparison')
OUT_DIR = BASE_OUT / 'cv_parts'
OUT_DIR.mkdir(parents=True, exist_ok=True)
hits = (glob.glob('/kaggle/input/**/pm10_prepared.pkl', recursive=True) +
        glob.glob('/kaggle/working/**/pm10_prepared.pkl', recursive=True))
if not hits:
    raise FileNotFoundError('فایل pm10_prepared.pkl را به‌عنوان Dataset به نوت‌بوک اضافه کنید.')
PREP_PATH = hits[0]
print('Prepared data:', PREP_PATH)
with open(PREP_PATH, 'rb') as f:
    prep = pickle.load(f)

if SKIP_EXISTING:
    previous = glob.glob(f'/kaggle/input/**/cv_{MODEL_NAME}_fold*.csv', recursive=True)
    for p in previous:
        dst = OUT_DIR / Path(p).name
        if not dst.exists():
            shutil.copy2(p, dst)
    if previous:
        print(f'{len(previous)} previous CSV file(s) copied to working output.')


In [ ]:
import math, os, time, pickle, json, glob, gc, random
from functools import partial
from pathlib import Path
import numpy as np, pandas as pd
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
from scipy.special import kv, gamma
from scipy.stats import norm

pyro.set_rng_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_default_dtype(torch.float64)

def log_ndtr(x):
    return torch.special.log_ndtr(x) if hasattr(torch.special,'log_ndtr') else torch.log(torch.clamp(0.5*(1+torch.erf(x/math.sqrt(2))),min=1e-300))

class TorchFCSN(dist.TorchDistribution):
    arg_constraints={}
    support=dist.constraints.real
    has_rsample=False
    def __init__(self,mu,sigma,C,Lambda,validate_args=None):
        self.mu,self.sigma,self.C,self.Lambda=mu,sigma,C,Lambda
        self.dim=mu.shape[-1]
        super().__init__(torch.Size(),torch.Size([self.dim]),validate_args=validate_args)
        L=torch.linalg.cholesky(C)
        C12=L.T
        I=torch.eye(self.dim,dtype=C.dtype,device=C.device)
        C12_inv=torch.linalg.solve_triangular(C12,I,upper=True)
        one=torch.ones(self.dim,dtype=C.dtype,device=C.device)
        delta=Lambda/torch.sqrt(1+Lambda**2)
        bd=math.sqrt(2/math.pi)*delta
        st=sigma/torch.sqrt(1-bd**2+1e-12)
        self.L,self.C12,self.one,self.bd,self.st=L,C12,one,bd,st
        self.mux=mu-(bd*st)*(one@C12)
        self.D=(Lambda/(st+1e-12))*C12_inv
        self.logdet_cov=self.dim*torch.log(st**2+1e-12)+2*torch.log(torch.diag(L)).sum()
        self.c1=st*Lambda/torch.sqrt(1+Lambda**2)
        self.c2=st/torch.sqrt(1+Lambda**2)
    def sample(self,sample_shape=torch.Size()):
        shape=sample_shape+torch.Size([self.dim])
        u1=torch.randn(shape,dtype=self.mu.dtype,device=self.mu.device)
        u2=torch.randn(shape,dtype=self.mu.dtype,device=self.mu.device)
        return self.mux+(self.c1*torch.abs(u2))@self.C12+(self.c2*u1)@self.C12
    def log_prob(self,x):
        if x.ndim==1: x=x.unsqueeze(0)
        diff=x-self.mux
        q=torch.linalg.solve_triangular(self.L,(diff/(self.st+1e-12)).T,upper=False).T
        logg=-0.5*(self.dim*math.log(2*math.pi)+self.logdet_cov+(q*q).sum(-1))
        z=(x-self.mu)@self.D+(self.bd*self.Lambda)*self.one
        return self.dim*math.log(2)+logg+log_ndtr(z).sum(-1)

def corr_matrix(knots,kind,phi=None,nu=1.5,jitter=1e-6):
    D=np.sqrt(((knots[:,None,:]-knots[None,:,:])**2).sum(2))
    if phi is None: phi=np.median(D[D>0])
    r=D/(phi+1e-12)
    if kind=='exponential':
        C=np.exp(-r)
    elif kind=='matern32':
        C=(1+math.sqrt(3)*r)*np.exp(-math.sqrt(3)*r)
    else: raise ValueError(kind)
    return C+jitter*np.eye(len(knots)),float(phi)

def make_tensors(prep,idx,C):
    return tuple(torch.tensor(np.asarray(a)[idx]) for a in [prep['y'],prep['X'],prep['B'],prep['G']])+(torch.tensor(C),)

def model(y,X,B,G,C,model_name):
    N,P=X.shape; K=B.shape[1]; J=G.shape[1]
    sigma_eps=pyro.sample('sigma_eps',dist.HalfNormal(5.0))
    sigma_theta=pyro.sample('sigma_theta',dist.HalfNormal(5.0))
    beta0=pyro.sample('beta0',dist.Normal(0,10))
    beta_rest=pyro.sample('beta_rest',dist.Normal(0,2).expand([P-1]).to_event(1))
    beta=torch.cat([beta0[None],beta_rest])
    zero=torch.zeros(K,dtype=y.dtype)
    L=torch.linalg.cholesky(C)
    if model_name!='gaussian':
        Lambda=pyro.sample('Lambda',dist.HalfNormal(2.0))
    th=[]
    for j in range(J):
        if model_name=='gaussian':
            thj=pyro.sample(f'theta_{j}',dist.MultivariateNormal(zero,scale_tril=sigma_theta*L))
        else:
            thj=pyro.sample(f'theta_{j}',TorchFCSN(zero,sigma_theta,C,Lambda))
        th.append(thj)
    Theta=torch.stack(th)
    mu=X@beta+torch.sum((B@Theta.T)*G,dim=1)
    pyro.sample('y',dist.Normal(mu,sigma_eps).to_event(1),obs=y)

def flatten_samples(samples):
    out={}
    for k,v in samples.items():
        a=v.detach().cpu().numpy()
        out[k]=a.reshape((-1,)+a.shape[2:]) if a.ndim>=2 else a.reshape(-1)
    return out

def posterior_mu(samples,X,B,G,J,draw_ids=None):
    X=np.asarray(X); B=np.asarray(B); G=np.asarray(G)
    S=len(samples['sigma_eps'])
    if draw_ids is None: draw_ids=np.arange(S)
    ans=[]
    for s in draw_ids:
        beta=np.r_[samples['beta0'][s],samples['beta_rest'][s]]
        Theta=np.stack([samples[f'theta_{j}'][s] for j in range(J)])
        ans.append(X@beta+np.sum((B@Theta.T)*G,axis=1))
    return np.asarray(ans)

def fit_one(prep,model_name,train_idx,warmup,draws,chains,target_accept,max_tree_depth,seed):
    kind='matern32' if model_name in ['proposed_matern_fscsn','gaussian'] else 'exponential'
    C,phi=corr_matrix(prep['knots'],kind)
    y,X,B,G,Ct=make_tensors(prep,train_idx,C)
    pyro.clear_param_store(); pyro.set_rng_seed(seed)
    kernel=NUTS(partial(model, model_name=model_name),
                target_accept_prob=target_accept,max_tree_depth=max_tree_depth)
    mcmc_kwargs=dict(warmup_steps=warmup,num_samples=draws,num_chains=chains,disable_progbar=False)
    if chains>1:
        mcmc_kwargs['mp_context']='fork'
    mcmc=MCMC(kernel,**mcmc_kwargs)
    t0=time.time(); mcmc.run(y,X,B,G,Ct); runtime=time.time()-t0
    grouped=mcmc.get_samples(group_by_chain=True)
    return grouped,dict(corr_kind=kind,phi_fixed=phi,runtime_sec=runtime)

In [ ]:
fold_labels = np.asarray(prep['fold'])
unique_folds = sorted(np.unique(fold_labels).tolist())
print('Available folds:', unique_folds)

settings = dict(
    warmup=100 if TEST_MODE else 700,
    draws=100 if TEST_MODE else 1500,
    chains=1 if TEST_MODE else 2,
    target_accept=0.95,
    max_tree_depth=10
)

all_rows = []
failed_folds = []

for FOLD in FOLDS_TO_RUN:
    print('\n' + '=' * 90)
    print(f'MODEL: {MODEL_NAME} | FOLD: {FOLD}')
    print('=' * 90)

    stem = f'cv_{MODEL_NAME}_fold{FOLD}'
    csv_path = OUT_DIR / f'{stem}.csv'
    pkl_path = OUT_DIR / f'{stem}.pkl'

      if SKIP_EXISTING and csv_path.exists():
        old = pd.read_csv(csv_path)
        if len(old) == 1:
            row = old.iloc[0].to_dict()
            all_rows.append(row)
            print('Skipped; existing result:', csv_path)
            print(row)
            continue

    train_idx = np.where(fold_labels != FOLD)[0]
    test_idx = np.where(fold_labels == FOLD)[0]
    print('Train/Test:', len(train_idx), len(test_idx))

    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError(f'Fold {FOLD} دارای مجموعه آموزش یا آزمون خالی است.')

    try:
        fold_seed = SEED + 100 * MODEL_INDEX + FOLD
        samples, meta = fit_one(
            prep, MODEL_NAME, train_idx,
            seed=fold_seed,
            **settings
        )

        flat = flatten_samples(samples)
        S = len(flat['sigma_eps'])
        use = min(S, 100 if TEST_MODE else 400)
        ids = np.linspace(0, S - 1, use).astype(int)

        MU = posterior_mu(
            flat,
            np.asarray(prep['X'])[test_idx],
            np.asarray(prep['B'])[test_idx],
            np.asarray(prep['G'])[test_idx],
            prep['meta']['J'],
            ids
        )
        ytest = np.asarray(prep['y'])[test_idx]
        sig = np.asarray(flat['sigma_eps'])[ids]

        pred = MU.mean(axis=0)
        rmse = float(np.sqrt(np.mean((ytest - pred) ** 2)))
        mae = float(np.mean(np.abs(ytest - pred)))

        rng = np.random.default_rng(fold_seed)
        YREP = MU + rng.normal(size=MU.shape) * sig[:, None]
        lo, hi = np.quantile(YREP, [.05, .95], axis=0)
        coverage = float(np.mean((ytest >= lo) & (ytest <= hi)))

        # Expected log predictive density روی داده‌های آزمون
        LL = (-0.5 * np.log(2 * np.pi * sig[:, None] ** 2)
              -0.5 * ((ytest[None, :] - MU) / sig[:, None]) ** 2)
        mx = LL.max(axis=0)
        elpd = float(np.sum(mx + np.log(np.mean(np.exp(LL - mx), axis=0))))

        row = dict(
            model=MODEL_NAME,
            fold=FOLD,
            n_train=len(train_idx),
            n_test=len(test_idx),
            RMSE=rmse,
            MAE=mae,
            coverage90=coverage,
            ELPD=elpd,
            eval_draws=use,
            **meta,
            **settings
        )
        print(row)

       
        pd.DataFrame([row]).to_csv(csv_path, index=False)
        with open(pkl_path, 'wb') as f:
            pickle.dump({
                'row': row,
                'test_idx': test_idx,
                'y_test': ytest,
                'pred_mean': pred,
                'pred_lo90': lo,
                'pred_hi90': hi
            }, f)
        print('Saved:', csv_path)
        all_rows.append(row)

        
        del samples, flat, MU, YREP, LL
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        failed_folds.append({'fold': FOLD, 'error': repr(exc)})
        print(f'ERROR in fold {FOLD}: {exc!r}')
        
        pd.DataFrame(failed_folds).to_csv(
            BASE_OUT / f'cv_failures_{MODEL_NAME}.csv', index=False
        )


csv_files = sorted(OUT_DIR.glob(f'cv_{MODEL_NAME}_fold*.csv'))
if not csv_files:
    raise RuntimeError('هیچ خروجی موفقی برای این مدل تولید نشد.')

results = pd.concat([pd.read_csv(p) for p in csv_files], ignore_index=True)
results = results.sort_values('fold').drop_duplicates('fold', keep='last')
results.to_csv(BASE_OUT / f'cv_all_folds_{MODEL_NAME}.csv', index=False)

metric_cols = ['RMSE', 'MAE', 'coverage90', 'ELPD', 'runtime_sec']
summary = {'model': MODEL_NAME, 'n_completed_folds': int(results['fold'].nunique())}
for col in metric_cols:
    summary[f'{col}_mean'] = float(results[col].mean())
    summary[f'{col}_sd'] = float(results[col].std(ddof=1)) if len(results) > 1 else np.nan
summary_df = pd.DataFrame([summary])
summary_df.to_csv(BASE_OUT / f'cv_summary_{MODEL_NAME}.csv', index=False)

print('\n' + '=' * 90)
print('PER-FOLD RESULTS')
print(results[['model', 'fold', 'n_train', 'n_test', 'RMSE', 'MAE', 'coverage90', 'ELPD', 'runtime_sec']].to_string(index=False))
print('\nSUMMARY ACROSS COMPLETED FOLDS')
print(summary_df.to_string(index=False))

if failed_folds:
    print('\nFailed folds:', failed_folds)
else:
    print('\nAll requested folds completed successfully.')

print('\nFinal files:')
print(BASE_OUT / f'cv_all_folds_{MODEL_NAME}.csv')
print(BASE_OUT / f'cv_summary_{MODEL_NAME}.csv')
